# Demo — Reading Quality Results & Choosing a Strategy

Quality results aren't a pass/fail score — each failure needs a *decision*: fix at the
**source**, fix **downstream** in the transform, or **quarantine** the bad records. This
walkthrough runs the suite, rolls up results by dimension, and demonstrates how to classify
one failure in code; the exercise asks you to triage all failing expectations and write a
prioritized remediation plan.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog

gc = GovernedCatalog("trailhead.db")
q = gc.run_quality_suite()
print("Failing expectations:", int((~q["success"]).sum()), "of", len(q))
q

## Roll up by dimension
Group by `dimension` to see where the platform is weakest across the six quality dimensions:

In [ ]:
q.groupby("dimension").agg(checks=("success", "size"),
                           failing=("success", lambda s: int((~s).sum())),
                           worst_pct=("unexpected_percent", "max")).reset_index()

## Inspect one failing expectation and assign a strategy
Filter to a single failure — `customer.email` on the **validity** dimension — and map it to one
of the three remediation paths (`"source"` | `"downstream"` | `"quarantine"`):

In [ ]:
email_fail = q[(q["table"] == "customer") & (q["column"] == "email")].copy()
sample_remediation = {"customer.email": "quarantine"}
email_fail["key"] = email_fail["table"] + "." + email_fail["column"]
email_fail["strategy"] = email_fail["key"].map(sample_remediation)
email_fail[["key", "dimension", "expectation_type", "unexpected_count", "unexpected_percent", "strategy"]]

**Why `quarantine` for `customer.email`?** Malformed addresses on a PII column create active
regulatory and operational harm — you cannot send transactional/marketing messages to `"n/a"`,
and returning corrupt contact records breaks GDPR subject-access and erasure workflows.
Quarantining isolates the bad rows immediately while the upstream producer fixes signup validation.

**Takeaway / your turn:** in the exercise, classify every failing `table.column` expectation as
`source`, `downstream`, or `quarantine`, and write a remediation plan prioritized by regulatory
severity.